In [34]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2", device_map="auto")

In [35]:
facts = [
    "The Earth orbits the Sun once approximately every 365.25 days.",
    "The Moon is Earth's only natural satellite.",
    "Jupiter is the largest planet in our solar system.",
    "Mars is often called the Red Planet because of iron oxide on its surface.",
    "Sunlight takes approximately eight minutes and twenty seconds to reach Earth.",
    "Water freezes at 0 degrees Celsius under standard atmospheric pressure.",
    "A water molecule contains two hydrogen atoms and one oxygen atom.",
    "Plants use sunlight to convert carbon dioxide and water into sugars through photosynthesis.",
    "The adult human skeleton typically contains 206 bones.",
    "The human heart has four chambers.",
    "Octopuses have three hearts.",
    "The blue whale is the largest animal known to have existed.",
    "Mount Everest is the highest mountain above sea level.",
    "The Pacific Ocean is the largest ocean on Earth.",
    "The Sahara is the largest hot desert on Earth.",
    "William Shakespeare wrote the play Hamlet.",
    "Leonardo da Vinci painted the Mona Lisa.",
    "Neil Armstrong became the first person to walk on the Moon in 1969.",
    "Guido van Rossum created the Python programming language.",
    "A triangle has three sides."
]

In [36]:
import json
from typing import List

import pandas as pd
import torch
import torch.nn.functional as F


def get_sentence_embedding(text: str | List[str]) -> torch.Tensor:
    """Return a normalized vector for one string, or a matrix for a batch of strings."""
    single_sentence = isinstance(text, str)
    sentences = [text] if single_sentence else text
    if not isinstance(sentences, list) or any(not isinstance(s, str) for s in sentences):
        raise TypeError("text must be a string or a list of strings")
    if not sentences:
        raise ValueError("Provide at least one sentence")

    model.eval()  # Disable dropout so repeated inputs give consistent embeddings.
    device = model.get_input_embeddings().weight.device
    inputs = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=256,  # MiniLM's sentence-embedding input limit.
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        token_embeddings = model(**inputs).last_hidden_state

        # Mean pooling: average real tokens, excluding padding positions.
        mask = inputs["attention_mask"].unsqueeze(-1).to(token_embeddings.dtype)
        pooled = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        # Unit-length vectors can later be compared using a dot product.
        normalized = F.normalize(pooled.float(), p=2, dim=1)

    # A single string returns shape (384,); a batch returns (batch_size, 384).
    return normalized[0] if single_sentence else normalized


def embed_documents(
    documents: List[str],
    csv_path: str = "document_embeddings.csv",
    batch_size: int = 32,
):
    """Return document/vector rows and save them to CSV with JSON-encoded vectors."""
    if not isinstance(documents, list) or any(not isinstance(doc, str) for doc in documents):
        raise TypeError("documents must be a list of strings")
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1")

    embeddings = []
    for start in range(0, len(documents), batch_size):
        batch_embeddings = get_sentence_embedding(documents[start:start + batch_size])
        embeddings.extend(batch_embeddings.cpu().tolist())

    result = pd.DataFrame({"document": documents, "embedding": embeddings})
    # CSV holds text, so encode each vector as JSON; read it back with json.loads.
    result.assign(embedding=result["embedding"].map(json.dumps)).to_csv(
        csv_path, index=False, encoding="utf-8"
    )
    return result






In [37]:
# Save relative to the notebook kernel's current working directory.
embedded_documents = embed_documents(facts, csv_path="document_embeddings.csv")
embedded_documents.head()

,document,embedding
0,The Earth orbits the Sun once approximately ev...,"[0.03695828840136528, 0.03904460370540619, 0.0..."
1,The Moon is Earth's only natural satellite.,"[0.07741774618625641, -0.018928850069642067, 0..."
2,Jupiter is the largest planet in our solar sys...,"[0.036511536687612534, -0.005018422845751047, ..."
3,Mars is often called the Red Planet because of...,"[0.06303933262825012, -0.039275914430618286, -..."
4,Sunlight takes approximately eight minutes and...,"[0.05287133529782295, 0.07307012379169464, 0.0..."


In [38]:
get_sentence_embedding(["This is a sentence","sentence"]).shape

torch.Size([2, 384])

In [39]:
def retrieve_documents(query: str, n_results: int = 1,csv_path : str = "document_embeddings.csv") -> List[str]:


    embedded_documents = pd.read_csv(csv_path, converters={"embedding": json.loads})
    """Return up to n_results document texts, ranked by descending similarity."""
    if n_results < 0:
        raise ValueError("n_results must be non-negative")
    if n_results == 0 or embedded_documents.empty:
        return []

    query_embed = get_sentence_embedding(query)
    document_embeddings = torch.tensor(
        embedded_documents["embedding"].tolist(),
        dtype=query_embed.dtype,
        device=query_embed.device,
    )

    # Each row is one document: matrix-vector multiplication computes all dot products.
    # Both query and document embeddings are normalized, so these are cosine similarities.
    scores = document_embeddings @ query_embed
    top_indices = torch.topk(scores, k=min(n_results, len(embedded_documents))).indices

    # Convert positional indices to document texts, highest score first.
    return embedded_documents.iloc[top_indices.cpu().tolist()]["document"].tolist()


In [40]:
retrieve_documents("how many chambers does the human heart have?")

['The human heart has four chambers.']

In [41]:
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen3-0.6B")
    

Device set to use cuda:0


In [42]:
import re


def strip_thinking(text: str) -> str:
    """Remove thinking blocks, including an unfinished block at the end."""
    return re.sub(r"<think>.*?(?:</think>|$)", "", text, flags=re.DOTALL).strip()


def generate_response(query:str,retrieved_documents):
    context = "\n".join(retrieved_documents)

    messages = [
        {"role": "system", "content": "Answer using the supplied context. If it does not contain the answer, say so."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]

    response = pipe(messages)
    for result in response:
        assistant_message = result["generated_text"][-1]
        assistant_message["content"] = strip_thinking(assistant_message["content"])

    return response

In [43]:
def pipeline_query(query: str):
    documents = retrieve_documents(query)

    
    response = generate_response(query,documents)

    return response[-1]["generated_text"][-1]["content"]

In [49]:
answer = pipeline_query("What is the red planet and why?")

answer

'The red planet is Mars, and it is called the Red Planet because of the presence of iron oxide on its surface.'